### Complete Loan Approval Prediction Pipeline (using ANN)

This section demonstrates the full process from data loading and preprocessing to training an ANN and making a prediction.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Load dataset
df = pd.read_csv("loan.csv")

# Remove unnecessary column
df.drop("Loan_ID", axis=1, inplace=True)

# Handle missing values
for col in df.columns:
    if df[col].dtype == 'object':
        df[col].fillna(df[col].mode()[0], inplace=True)
    else:
        df[col].fillna(df[col].median(), inplace=True)

# Convert target column
df['Loan_Status'] = df['Loan_Status'].map({
    'Y': 1,
    'N': 0
})

# One-Hot Encoding
categorical_cols = [
    'Gender',
    'Married',
    'Education',
    'Self_Employed',
    'Property_Area'
]

df = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True
)

# Dependents
df['Dependents'] = df['Dependents'].replace('3+', 3)
df['Dependents'] = df['Dependents'].astype(int)

# Features and Target
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Feature Scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Data Preprocessing Complete. Shapes:")
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")

Data Preprocessing Complete. Shapes:
X_train: (491, 12)
X_test: (123, 12)


/tmp/ipykernel_1546/2962423167.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)
/tmp/ipykernel_1546/2962423167.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try

#### Define, Compile, and Train the ANN Model

In [3]:
# Define the ANN model
ann_model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid') # Output layer for binary classification
])

# Compile the model
ann_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("ANN Model Summary:")
ann_model.summary()

# Train the model
print("\nTraining ANN Model...")
history = ann_model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=0)
print("ANN Model Training Complete.")

ANN Model Summary:


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)


Training ANN Model...
ANN Model Training Complete.


#### Evaluate the ANN Model

In [4]:
# Evaluate the model on the test set
loss, accuracy = ann_model.evaluate(X_test, y_test, verbose=0)
print(f"\nANN Test Accuracy: {accuracy:.4f}")

# Make predictions (probabilities)
y_pred_ann_prob = ann_model.predict(X_test)
# Convert probabilities to binary class labels (0 or 1)
y_pred_ann = (y_pred_ann_prob > 0.5).astype(int)

print("\nConfusion Matrix for ANN:\n", confusion_matrix(y_test, y_pred_ann))
print("\nClassification Report for ANN:\n", classification_report(y_test, y_pred_ann))


ANN Test Accuracy: 0.7886
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step

Confusion Matrix for ANN:
 [[18 25]
 [ 1 79]]

Classification Report for ANN:
               precision    recall  f1-score   support

           0       0.95      0.42      0.58        43
           1       0.76      0.99      0.86        80

    accuracy                           0.79       123
   macro avg       0.85      0.70      0.72       123
weighted avg       0.83      0.79      0.76       123



#### Example Prediction: Is a loan approved or not?

In [8]:
# Let's create a hypothetical new applicant's data for prediction
# This needs to match the columns and order of X before one-hot encoding
# Example values (replace with actual new applicant data)
new_applicant_data = pd.DataFrame([
    {
        'Gender': 'Male',
        'Married': 'Yes',
        'Dependents': 1,
        'Education': 'Graduate',
        'Self_Employed': 'No',
        'ApplicantIncome': 5000,
        'CoapplicantIncome': 1000,
        'LoanAmount': 120,
        'Loan_Amount_Term': 360,
        'Credit_History': 1,
        'Property_Area': 'Semiurban'
    }
])

# The original fillna loop is removed as new_applicant_data in this example has no missing values
# and 'df' no longer contains original categorical columns for imputation.
# If new_applicant_data might have missing values, a more robust imputation logic
# (e.g., using modes/medians stored from the *training* df before one-hot encoding)
# would be needed here.

# 2. Dependents conversion
new_applicant_data['Dependents'] = new_applicant_data['Dependents'].replace('3+', 3)
new_applicant_data['Dependents'] = new_applicant_data['Dependents'].astype(int)

# 3. One-Hot Encoding for new_applicant_data
# Apply get_dummies directly to the new_applicant_data
new_applicant_dummies = pd.get_dummies(new_applicant_data, columns=categorical_cols, drop_first=True)

# Ensure the new applicant data has the exact same columns as the training data (X)
# Use reindex to add missing columns (with fill_value=0) and drop any extra ones
new_applicant_processed = new_applicant_dummies.reindex(columns=X.columns, fill_value=0)

# 4. Scale the new applicant data using the *fitted* scaler
new_applicant_scaled = scaler.transform(new_applicant_processed)

# Make prediction with the ANN model
prediction_prob = ann_model.predict(new_applicant_scaled)[0][0]
prediction_label = 1 if prediction_prob > 0.5 else 0

print(f"\nPrediction Probability for New Applicant: {prediction_prob:.4f}")
if prediction_label == 1:
    print("Loan Status: Approved (Prediction: Y)")
else:
    print("Loan Status: Not Approved (Prediction: N)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step

Prediction Probability for New Applicant: 0.7370
Loan Status: Approved (Prediction: Y)
